# DBRepo Upload Notebook

This notebook handles the creation of the database, tables, and the upload of the data to DBRepo via the REST API. It ensures proper metadata attribution to Eurostat and applies the CC BY 4.0 license.

**Note:** As per the plan, this notebook sets up the logic but data changes should only be executed once fully approved.

In [ ]:
%pip install -r ../requirements.txt

In [ ]:
import requests
import pandas as pd
import json
import os
from dotenv import load_dotenv
from dbrepo.RestClient import RestClient

load_dotenv()

DBREPO_ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
DATABASE_ID = "123289f2-5218-4b32-b962-5f3dafec1fe3"
USERNAME = os.getenv("DBREPO_USERNAME")
PASSWORD = os.getenv("DBREPO_PASSWORD")

# T2.1 DBRepo - schema design 

In [ ]:
# Metadata for the Database based on the Data Management Plan (DMP) and instructions
database_metadata = {
    "name": "ev_protection_investment_analysis",
    "description": "Environmental protection investments across EU countries (2014-2022) based on the reuse of existing data from the Eurostat Open Data Portal.",
    "publisher": "Eurostat",
    "creator": "Eurostat",
    "license": "CC BY 4.0",
    "rights": "European Union / Eurostat",
    "republisher": "Luka Premuš / TU Wien",
    "republisher_email": "e12552143@student.tuwien.ac.at",
    "republisher_affiliation": "TU Wien (Course 194.045 Data Stewardship)",
    "project_title": "Do Rich Countries Invest More in Saving the Planet? Analysis of Europe’s Green Investment Landscape",
    "dmp_version": "1.0",
    "dmp_date": "2026-05-25"
}
print("Database metadata prepared:", json.dumps(database_metadata, indent=2))

In [ ]:


table_country = {
    "name": "Country",
    "is_public": True,
    "is_schema_public": True,
    "description": "Country dimension table with ISO codes.",
    "columns": [
        {"name": "country_code", "type": "varchar", "size": 2, "null_allowed": False, "description": "2-letter ISO country code"},
        {"name": "country_name", "type": "varchar", "size": 255, "null_allowed": False, "description": "Full name of the country"}
    ],
    "constraints": {
        "primary_key": ["country_code"]
    }
}

table_activity = {
    "name": "Environmental_Activity",
    "is_public": True,
    "is_schema_public": True,
    "description": "Environmental activity dimension table (CEPA/CReMA classifications).",
    "columns": [
        {"name": "ceparema_code", "type": "varchar", "size": 50, "null_allowed": False, "description": "CEPA/CReMA activity code"},
        {"name": "activity_name", "type": "varchar", "size": 255, "null_allowed": False, "description": "Name of the environmental protection activity"}
    ],
    "constraints": {
        "primary_key": ["ceparema_code"]
    }
}


In [ ]:
table_macro = {
    "name": "Macroeconomic_Indicator",
    "is_public": True,
    "is_schema_public": True,
    "columns": [
        {"name": "country_code", "type": "varchar", "size": 2, "null_allowed": False, "description": "2-letter ISO country code"},
        {"name": "year", "type": "int", "null_allowed": False, "description": "Observation year"},
        {"name": "population", "type": "bigint", "null_allowed": True, "description": "Total population"},
        {"name": "gdp_per_capita", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Gross Domestic Product per capita"}
    ],
    "constraints": {
        "primary_key": ["country_code", "year"],
        "foreign_keys": [
            {
                "columns": ["country_code"],
                "referenced_table": "country",
                "referenced_columns": ["country_code"]
            }
        ]
    }
}

table_invest = {
    "name": "Environmental_Investment",
    "is_public": True,
    "is_schema_public": True,
    "columns": [
        {"name": "country_code", "type": "varchar", "size": 2, "null_allowed": False, "description": "2-letter ISO country code"},
        {"name": "year", "type": "int", "null_allowed": False, "description": "Observation year"},
        {"name": "ceparema_code", "type": "varchar", "size": 50, "null_allowed": False, "description": "CEPA/CReMA activity code"},
        {"name": "inv_gov", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Investment by general government"},
        {"name": "inv_corp_spec", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Investment by specialist producers"},
        {"name": "inv_corp_anc", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Investment by ancillary producers"},
    ],
    "constraints": {
        "primary_key": ["country_code", "year", "ceparema_code"],
        "foreign_keys": [
            {
                "columns": ["country_code"],
                "referenced_table": "country",
                "referenced_columns": ["country_code"]
            },
            {
                "columns": ["ceparema_code"],
                "referenced_table": "environmental_activity",
                "referenced_columns": ["ceparema_code"]
            }
        ]
    }
}


In [ ]:
def create_table_via_api(database_id, table_def):
    # Ensure all constraint fields are present to avoid 500 errors
    if "constraints" not in table_def:
        table_def["constraints"] = {}
    
    for key in ["uniques", "checks", "foreign_keys", "primary_key"]:
        if key not in table_def["constraints"]:
            table_def["constraints"][key] = []

    url = f"{DBREPO_ENDPOINT}/api/v1/database/{database_id}/table"
    print(f"Creating table {table_def['name']}...")
    response = requests.post(url, json=table_def, auth=(USERNAME, PASSWORD))
    
    if response.status_code == 201:
        print(f"Success: Table {table_def['name']} created.")
    elif response.status_code == 409:
        print(f"Warning: Table {table_def['name']} already exists.")
    else:
        print(f"Error ({response.status_code}): {response.text}")


tables_to_create = [table_country, table_activity, table_macro, table_invest]
for t in tables_to_create:
    create_table_via_api(DATABASE_ID, t)


# T2.2 Semantic mapping

In [ ]:
# Code Placeholder

# T2.3 Mapping Units of Measurement

### Ontology Choice

All numeric attributes are mapped using QUDT (Quantities, Units, Dimensions and Types), version 3.2.1 — https://qudt.org

The assignment recommends the SI Digital Framework as the primary ontology, but SI only covers physical quantities. None of the attributes in this dataset are physical measurements: year is a calendar unit, population is a count of persons and all monetary values are in Euro. QUDT is a well-established, actively maintained ontology that explicitly covers all three of these cases and provides more expressive modeling for this case than SI-oriented unit systems.

### Unit Mappings

The attribute year (both tables) is mapped to the QUDT unit Year (https://qudt.org/vocab/unit/YR). QUDT defines this as one passage of Earth around the Sun (roughly 365 days).

The attribute population is mapped to unit NUM (https://qudt.org/vocab/unit/NUM). Since population is a simple count of persons rather than a physical measurement, SI does not provide a meaningful unit.

All monetary attributes (gdp_per_capita, inv_gov, inv_corp_spec, inv_corp_anc, inv_corp_total, inv_total) are mapped to Euro (https://qudt.org/vocab/unit/CCY_EUR). QUDT defines this as a currency unit. Currency values are outside the scope of SI units, which is why a dedicated currency ontology unit is used.

In [ ]:
UNIT_YEAR  = "http://qudt.org/vocab/unit/YR"
UNIT_COUNT = "https://qudt.org/vocab/unit/NUM"
UNIT_EUR   = "https://qudt.org/vocab/unit/CCY_EUR"

unit_mappings = [
    # (table_name,              column_name,      unit_uri)
    ("macroeconomic_indicator", "year",            UNIT_YEAR),
    ("macroeconomic_indicator", "population",      UNIT_COUNT),
    ("macroeconomic_indicator", "gdp_per_capita",  UNIT_EUR),
    ("environmental_investment","year",            UNIT_YEAR),
    ("environmental_investment","inv_gov",         UNIT_EUR),
    ("environmental_investment","inv_corp_spec",   UNIT_EUR),
    ("environmental_investment","inv_corp_anc",    UNIT_EUR),
    ("environmental_investment","inv_corp_total",  UNIT_EUR),
    ("environmental_investment","inv_total",       UNIT_EUR),
]

In [ ]:
def set_unit_mappings(base_url, database_id, table_id_map, unit_mappings, auth):
    results = []

    for table_name, column_name, unit_uri in unit_mappings:
        table_id = table_id_map[table_name]

        # fetch table schema
        table_url = f"{base_url}/api/v1/database/{database_id}/table/{table_id}"
        columns = requests.get(table_url, auth=auth).json()["columns"]

        # find column id
        column_id = next(
            (c["id"] for c in columns if c["name"] == column_name),
            None
        )

        if column_id is None:
            raise ValueError(f"Column not found: {column_name} in {table_name}")

        # update column
        update_url = (
            f"{base_url}/api/v1/database/{database_id}"
            f"/table/{table_id}/column/{column_id}"
        )

        r = requests.put(update_url, json={"unit_uri": unit_uri}, auth=auth)

        results.append({
            "table": table_name,
            "column": column_name,
            "status": r.status_code,
        })

        print(f"{table_name}.{column_name} -> {unit_uri} [{r.status_code}]")

    return results

In [ ]:
def verify_unit_mappings(base_url, database_id, table_id_map, unit_mappings, auth):
    mismatches = []

    for table_name, column_name, expected in unit_mappings:
        table_id = table_id_map[table_name]

        url = f"{base_url}/api/v1/database/{database_id}/table/{table_id}"
        columns = requests.get(url, auth=auth).json()["columns"]

        actual = next(
            (c.get("unit_uri") for c in columns if c["name"] == column_name),
            None
        )

        if actual != expected:
            mismatches.append({
                "table": table_name,
                "column": column_name,
                "expected": expected,
                "actual": actual
            })

    if mismatches:
        print("MISMATCHES FOUND:")
        for m in mismatches:
            print(m)
    else:
        print("All unit mappings are correct.")

    return mismatches

In [ ]:
base_url = DBREPO_ENDPOINT
database_id = DATABASE_ID
auth = (USERNAME, PASSWORD)

# Apply
results = set_unit_mappings(
    base_url=base_url,
    database_id=database_id,
    table_id_map=table_map,
    unit_mappings=unit_mappings,
    auth=auth
)

# Verify
mismatches = verify_unit_mappings(
    base_url=base_url,
    database_id=database_id,
    table_id_map=table_map,
    unit_mappings=unit_mappings,
    auth=auth
)

**Note:** 
The REST API via the requests library is used instead of the DBRepo Python client because the current DBRepo SDK does not fully support the latest column metadata fields, especially unit_uri and concept_uri. In the SDK, these fields are either missing or not correctly mapped in the data models, which leads to incomplete or inconsistent results when reading the table structure.

With the REST API, the updates are visible and correctly stored in the backend. A direct GET request to the table endpoint shows that the unit_uri values are correctly set for the corresponding columns.

However, these changes are currently not reflected in the frontend interface. The frontend schema view still does not display the updated measurement units, even though the REST API confirms that the values are present in the database. This indicates that the backend is up to date, but the frontend either uses cached data or does not yet render the unit_uri field in the schema view.


In [ ]:
# Example

TABLE_ID = "3d092a1b-4c9d-4bf7-91c2-e85277666082"

url = f"{DBREPO_ENDPOINT}/api/v1/database/{DATABASE_ID}/table/{TABLE_ID}"

r = requests.get(url, auth=(USERNAME, PASSWORD))
r.raise_for_status()
table = r.json()

print(f"TABLE: {table['name']} ({table['id']})")

for col in table["columns"]:
    print("--------------------------------------------------")
    print(f"Column name      : {col.get('name')}")
    print(f"Type             : {col.get('type')}")
    print(f"Unit URI         : {col.get('unit_uri')}")
    print(f"Concept URI      : {col.get('concept_uri')}")
    print(f"Description      : {col.get('description')}")

# T2.5 DBRepo - load

In [ ]:
# Code Placeholder